In [6]:
from glob import glob
import os
from tqdm import tqdm
from PIL import Image
from torchvision import transforms
import torch
from skimage import morphology as morph

In [7]:
crop_size = 64
split = "train"   # choisir train, val ou test

root_dir = f"../data/dataset_crops_paires/{split}"
os.makedirs(f"{root_dir}/sat", exist_ok=True)   # Création du dossier pour les images satellites si besoin
os.makedirs(f"{root_dir}/burn", exist_ok=True)   # Création du dossier pour les annotations si besoin

In [8]:
def imgs_to_niveaucouleur(imgs):
    imgs = imgs.unsqueeze(0)   # pour rajouter la dimension B quand on appelle depuis le loader et pas dans la boucle d'entraînement

    imgs = (imgs * 255).byte()
    imgs = imgs.permute(1, 0, 2, 3)   # (B, C, H, W) -> (C, B, H, W)

    colors = [[181, 254, 142], [254, 217, 142], [254, 153, 41], [204, 76, 2]]

    masks = []
    for c in colors:
        color_tensor = torch.tensor(c, dtype=imgs.dtype, device=imgs.device).view(-1, 1, 1, 1)
        mask = (imgs == color_tensor).all(dim=0)
        masks.append(mask)


    # # Combiner tous les masks
    # combined_mask = masks[0].clone()
    # for m in masks[1:]:
    #     combined_mask |= m  # ou “torch.logical_or(combined_mask, m)”

    # # Vérifier les pixels qui ne correspondent à aucune couleur
    # unknown_pixels = ~combined_mask
    # n_unknown = unknown_pixels.sum().item()

    # print(f"Nombre de pixels inconnus : {n_unknown}")


    # appliquer un filtrage (ouverture puis fermeture (voir morphologie mathématique))
    filtered_from_masks = []
    for mask in masks:
        filtered_mask = morph.area_opening(mask.squeeze(0).cpu().numpy(), area_threshold=32)
        filtered_mask = torch.from_numpy(filtered_mask).to(imgs.device)
        difference_mask = (mask != filtered_mask).bool()
        filtered_from_masks.append(difference_mask)

    union_maps = torch.zeros_like(masks[0], dtype=torch.bool, device=imgs.device)
    for m in filtered_from_masks:
        union_maps |= m

    masks.append(union_maps)


    out = torch.zeros((imgs.shape[1], imgs.shape[2], imgs.shape[3]), dtype=torch.int64, device=imgs.device)
    for i, mask in enumerate(masks):
        out[mask] = i + 1

    out = out.squeeze(0)  # (1, H, W) -> (H, W)

    return out

In [9]:
# get all image paths
img_paths = []

for path in glob(os.path.join(f"../data/dataset/{split}", "**", f"*S2L2A*"), recursive=True):
    if path.lower().endswith((".jpg", ".jpeg", ".png")):

        # chemin vers l'image satellite
        img_path = path

        # chemin vers le masque de brûlé
        path_to_img_gra = img_path.replace("S2L2A", "GRA")
        path_to_img_del = img_path.replace("S2L2A", "DEL")
        if os.path.exists(path_to_img_gra):
            img_path_burn = path_to_img_gra
        elif os.path.exists(path_to_img_del):
            img_path_burn = path_to_img_del
        else:
            print("********** image brûlée non trouvée *********")
            continue

        img_paths.append((img_path, img_path_burn))


print(len(img_paths), "images trouvées")

281 images trouvées


In [10]:
idx = 0

for i, (img_path, img_path_burn) in tqdm(enumerate(img_paths), total=len(img_paths)):

    img = Image.open(img_path).convert("RGB")
    img_burn = Image.open(img_path_burn).convert("RGB")
    assert img.size == img_burn.size, f"Image size mismatch: {img_path} and {img_path_burn}"


    width, height = img.size

    # Calculer combien de crops on peut faire horizontalement et verticalement
    n_crops_x = width // crop_size
    n_crops_y = height // crop_size

    # Découper en crops 32x32
    for i in range(n_crops_y):
        for j in range(n_crops_x):
            left = j * crop_size
            upper = i * crop_size
            right = left + crop_size
            lower = upper + crop_size

            crop = img.crop((left, upper, right, lower))
            crop_burn = img_burn.crop((left, upper, right, lower))

            # traiter les images pour obtenir des valeurs de classes et pour enlever les composantes connexes trop petites
            crop_burn = transforms.ToTensor()(crop_burn)
            crop_burn = imgs_to_niveaucouleur(crop_burn)
            crop_burn = crop_burn.cpu().numpy().astype("uint8")
            crop_burn = Image.fromarray(crop_burn)
            crop_burn = crop_burn.convert("L")

            crop.save(f"{root_dir}/sat/crop_{idx}.png")
            crop_burn.save(f"{root_dir}/burn/crop_{idx}.png")

            idx += 1


100%|██████████| 281/281 [54:27<00:00, 11.63s/it]  
